# Argus — CNN Training (Face Crops)

A small CNN trained directly on cropped RGB face images — no hand-engineered features at all.
Like `04_random_forest_training.ipynb` and `05_dense_nn_training.ipynb`, this is a single-image-
in, single-label-out model: one cropped face predicts one drowsiness level, no sequence, no
buffer across frames.

**Reads:** `dataset_processed/face_crops_index.csv` and the `.jpg` files it points to, written by
[`06_dataset_creation_face_crops.ipynb`](./06_dataset_creation_face_crops.ipynb) — run that first.

**Writes:** `models/cnn_face_crops_<VERSION>.keras`.

**Why "tiny."** The other three model families (LSTM, RandomForest, Dense NN) all work from a
59-dimensional-or-smaller hand-engineered feature vector. This model instead has to learn visual
features from raw pixels, which is a fundamentally harder learning problem to solve from the
amount of data this project has (tens of subjects, not the thousands a large vision model would
want) — so the architecture below is deliberately small and heavily regularized (few conv
blocks, dropout, no pretrained backbone) rather than a scaled-up architecture that would likely
just overfit. This is also the only model family with a plausible path to running its own
inference directly on cropped camera frames without the geometric-feature pipeline at all, which
is worth noting for the titulación report as a architectural alternative, even though the project
is standardizing on the LSTM for deployment (see `03_model_training_lstm.ipynb`'s Design
Decision cell) — this notebook exists to have that comparison on record, not to replace it.


## Setup

Mounts Drive and defines the paths/constants this notebook needs. Independent of any other notebook's in-memory state — only depends on `face_crops_index.csv` and the images it references.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_folder = "/content/drive/MyDrive/Argus"
print(f"Google Drive successfully mounted! Base project directory: {project_folder}")


In [ ]:
import os
import pandas as pd

# --- Argus project paths (must match 06_dataset_creation_face_crops.ipynb) ---
models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
processed_folder = f"{dataset_folder}/dataset_processed"
face_crops_index_csv_path = os.path.join(processed_folder, "face_crops_index.csv")

if not os.path.exists(face_crops_index_csv_path):
    raise FileNotFoundError(
        f"'{face_crops_index_csv_path}' not found. Run 06_dataset_creation_face_crops.ipynb "
        "first — this notebook only reads the dataset it produces, it doesn't build it."
    )

df_face_crops = pd.read_csv(face_crops_index_csv_path)
print(f"Loaded face crop index: {len(df_face_crops)} images, "
      f"{df_face_crops['subject'].nunique()} subjects.")


In [ ]:
import joblib
import datetime

# --- Model Management Configuration ---
FORCE_RETRAIN = True

VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")
cnn_model_path = os.path.join(models_folder, f"cnn_face_crops_{VERSION_STR}.keras")

def get_latest_model(folder, prefix, extension):
    """Return the path of the most recently versioned model matching prefix/extension, or None."""
    if not os.path.exists(folder):
        return None
    files = [f for f in os.listdir(folder) if f.startswith(prefix) and f.endswith(extension)]
    if not files:
        return None
    return os.path.join(folder, sorted(files)[-1])

print(f"✅ Model management initialized. Current version: {VERSION_STR}. Force retrain: {FORCE_RETRAIN}")


## Splitting the Dataset

Group-aware 80/20 split by subject, same rationale as every other notebook: crops from the same
clip are highly correlated (near-duplicate faces a few frames apart), so a plain random split
would leak near-duplicates across train/test and inflate reported accuracy.

The split happens on the **index CSV** (file paths + labels), before any image is decoded — the
actual pixel loading happens lazily in the `tf.data` pipeline below.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_face_crops, df_face_crops['level'], groups=df_face_crops['subject']))

df_train = df_face_crops.iloc[train_idx].reset_index(drop=True)
df_test = df_face_crops.iloc[test_idx].reset_index(drop=True)

print(f"Train set size: {len(df_train)} crops ({df_train['subject'].nunique()} subjects)")
print(f"Test set size: {len(df_test)} crops ({df_test['subject'].nunique()} subjects)")
print("Subjects are disjoint between training and testing sets (group-aware split).")


## Building the `tf.data` Pipeline

Crops are variable-size (see `06_dataset_creation_face_crops.ipynb`), so every image is resized
to a fixed `IMG_SIZE x IMG_SIZE` here. `IMG_SIZE = 96` is chosen as a small-but-workable input
resolution for a *tiny* CNN — large enough to keep eye/mouth detail legible after the face
detector's crop, small enough to keep the model and its memory footprint modest. Pixel scaling
(`1/255`) is done as a `Rescaling` layer inside the model itself (see the model definition below)
rather than as a separate `StandardScaler` artifact, so the trained `.keras` file is a fully
self-contained image-in, prediction-out artifact with no companion scaler file to keep in sync.


In [ ]:
!pip install -q albumentations

import albumentations as A
import numpy as np
import tensorflow as tf

# Lighting/effects-only transforms for training-time augmentation -- no geometric distortion
# (crop/rotate/shear) here, consistent with the flip-only geometric augmentation applied
# separately below, since aggressive geometric transforms risk warping the drowsiness-relevant
# eye/mouth region. Kept mild, in the same spirit as the plain tf.image brightness jitter this
# replaces: RandomBrightnessContrast/RandomGamma/HueSaturationValue cover a wider range of
# realistic lighting variation (dashboard glare, shadow, skin-tone shift under cabin lighting)
# than a single brightness delta did.
_lighting_aug = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.3),
])

def _albumentations_transform(image_np):
    # Albumentations' pixel-level transforms expect uint8 images; the tf.data pipeline above
    # hands us float32 in [0, 255] (Rescaling to [0, 1] happens inside the model, not here).
    augmented = _lighting_aug(image=image_np.astype(np.uint8))['image']
    return augmented.astype(np.float32)

def apply_lighting_augmentation(image):
    # Albumentations isn't a native TF op, so it has to run outside the graph via
    # tf.numpy_function; that erases static shape info, hence the explicit set_shape after.
    aug_image = tf.numpy_function(func=_albumentations_transform, inp=[image], Tout=tf.float32)
    aug_image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return aug_image

print("✅ Albumentations lighting/effects augmentation ready.")


In [ ]:
import tensorflow as tf

IMG_SIZE = 96
BATCH_SIZE = 32
CLASS_NAMES = ['Alert', 'Low Vigilant', 'Drowsy']

def load_and_preprocess(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

def make_dataset(df, training: bool):
    paths = df['image_path'].to_numpy()
    labels = (df['level'].to_numpy(dtype=np.int32) - 1)  # 0-indexed: 0=Alert, 1=Low Vigilant, 2=Drowsy
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=2048, seed=42)
        # Light augmentation: crops are already tightly framed on the face, so augmentation stays
        # mild (small flip + Albumentations lighting/effects) rather than aggressive geometric
        # distortion that could distort the drowsiness-relevant eye/mouth region. The brightness-
        # only tf.image jitter this used to do is now covered (and extended) by
        # apply_lighting_augmentation, defined above.
        ds = ds.map(lambda img, lbl: (tf.image.random_flip_left_right(img), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(lambda img, lbl: (apply_lighting_augmentation(img), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(df_train, training=True)
test_ds = make_dataset(df_test, training=False)

print(f"IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}")


## Tiny CNN Model Definition

Three convolutional blocks (`Conv2D -> BatchNorm -> MaxPool`), increasing filter count, followed
by `GlobalAveragePooling2D` (instead of `Flatten` + a large `Dense` layer, to keep parameter
count down and reduce overfitting risk) and a small classification head.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Rescaling, Conv2D, BatchNormalization, MaxPooling2D,
    GlobalAveragePooling2D, Dense, Dropout,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

num_classes = len(CLASS_NAMES)

model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Rescaling(1. / 255),

    Conv2D(16, 3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(32, 3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(64, 3, padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    GlobalAveragePooling2D(),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(num_classes, activation='softmax'),
])

optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## CNN Model Training

`class_weight`, computed from the training set's actual label distribution, for the same reason
as every other model here: `Drowsy` is plausibly the rarest class and the most safety-critical
one to not neglect.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
checkpoint = ModelCheckpoint(filepath=os.path.join(models_folder, 'best_cnn_face_crops.keras'), monitor='val_accuracy', save_best_only=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1)

train_labels = (df_train['level'].to_numpy(dtype=np.int32) - 1)
weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
weight_dict = dict(zip(np.unique(train_labels).tolist(), weights))

print("🚀 Training CNN...")
history = model.fit(
    train_ds,
    epochs=100,
    validation_data=test_ds,
    callbacks=[early_stopping, checkpoint, reduce_lr],
    class_weight=weight_dict,
    verbose=1,
)

model.save(cnn_model_path)
print(f"💾 Model saved: {cnn_model_path}")


## CNN Model Evaluation


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"\nCNN Test Loss: {loss:.4f}")
print(f"CNN Test Accuracy: {accuracy:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report (CNN):")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (CNN)')
plt.show()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()


### Reading this result alongside the other models

This model has strictly less information available per prediction than the geometric-feature
models — it never sees EAR/MAR/blendshapes/pose directly, only pixels, and has to (re-)learn
whatever of that signal it can from a comparatively small image dataset. Treat any accuracy gap
against RandomForest/Dense NN as evidence about how hard that visual-feature-learning problem is
from this dataset's size, not as a verdict on face-crop CNNs as an approach in general — a much
larger face dataset (or a pretrained backbone) would likely close much of that gap, neither of
which this project's data budget currently supports.
